In [11]:
#word opcunt using streaming
from pyspark import SparkContext
from pyspark.streaming import StreamingContext
import threading
import time
import socket



In [12]:
#craete spark context

sc= SparkContext("local[2]","InbuiltDatasetStreaming")
ssc= StreamingContext(sc,3) #batch interval 3 sec

/usr/local/lib/python3.12/dist-packages/pyspark/streaming/context.py:72: FutureWarning: DStream is deprecated as of Spark 3.4.0. Migrate to Structured Streaming.
  warnings.warn(


In [13]:
#streaming source
lines=ssc.socketTextStream("localhost",9999)

In [14]:
#processing logic
words=lines.flatMap(lambda line: line.split(" "))
pairs=words.map(lambda word: (word,1))
word_counts=pairs.reduceByKey(lambda a,b: a+b)
word_counts.pprint()

In [15]:
word_counts.saveAsTextFiles("stream_output/wordcount")

In [16]:
#start steaming job
def start_straeming():
  ssc.start()
  ssc.awaitTermination(60)
  ssc.stop(stopSparkContext=True,stopGraceFully=True)

In [17]:
#simultate streamniumgn with inbuuilt datset
def send_data():
  from sklearn.datasets import fetch_20newsgroups
  dataset = fetch_20newsgroups(subset='train',remove=('headers','footers','qutoes'))
  data=dataset.data[:50]
  server=socket.socket(socket.AF_INET,socket.SOCK_STREAM)
  server.bind(("localhost",9999))
  server.listen(1)
  print("socket server started on port 9999...")
  conn, addr=server.accept()

  for doc in data:
    line=doc.split("\n")[0]
    if line.strip():
      conn.send((line + "\n").encode("utf-8"))
      time.sleep(2)
  conn.close()
  server.close()

In [18]:
#runnig everything
threading.Thread(target=send_data).start()
start_straeming()

Exception in thread Thread-11 (send_data):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipython-input-641975022.py", line 7, in send_data
OSError: [Errno 98] Address already in use


-------------------------------------------
Time: 2025-09-27 07:49:33
-------------------------------------------
('I', 2)
('wondering', 1)
('anyone', 1)
('could', 1)
('me', 1)
('this', 1)
('car', 1)
('saw', 1)
('was', 1)
('if', 1)
...

-------------------------------------------
Time: 2025-09-27 07:49:36
-------------------------------------------
('A', 1)
('fair', 1)
('number', 1)
('of', 1)
('souls', 1)
('upgraded', 1)
('their', 1)
('clock', 1)
('have', 1)
('brave', 1)
...

-------------------------------------------
Time: 2025-09-27 07:49:39
-------------------------------------------
('well', 1)
('plus', 1)
('finally', 1)
('up', 1)
('this', 1)
('Robert', 1)
('J.C.', 1)
('folks,', 1)
('my', 1)
('mac', 1)
...

-------------------------------------------
Time: 2025-09-27 07:49:42
-------------------------------------------
('From', 1)
('<C5owCB.n3p@world.std.com>,', 1)
('by', 1)
('tombaker@world.std.com', 1)
('A', 1)
('Baker):', 1)
('article', 1)
('(Tom', 1)

-------------------------

Exception in thread Thread-9 (send_data):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipython-input-641975022.py", line 15, in send_data
BrokenPipeError: [Errno 32] Broken pipe


-------------------------------------------
Time: 2025-09-27 07:50:36
-------------------------------------------

-------------------------------------------
Time: 2025-09-27 07:50:39
-------------------------------------------

